In [1]:
import duckdb
import numpy as np
import pandas as pd

prep = duckdb.sql("SELECT * FROM read_parquet('../data/gold/nyc/listings_prepared.parquet')").df()
comps = duckdb.sql("SELECT * FROM read_parquet('../data/gold/nyc/comps.parquet')").df()

print("Prepared listings:", len(prep))
print("Listings with comps:", len(comps))
print("Priced listings:", prep["base_price"].notna().sum())
print("Entire homes still missing bedrooms:",
      (prep["bedrooms_missing"] & (prep["room_type"] == "Entire home/apt")).sum())

Prepared listings: 30259
Listings with comps: 21514
Priced listings: 21514
Entire homes still missing bedrooms: 860


In [2]:
# Do the comparison numbers make sense?
print(comps["gap_vs_median_pct"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(1))
print()

# A listing priced above its comp median should be more expensive than most comps
above = comps[comps["gap_vs_median_pct"] > 0]
print("Above median but fewer than 40% of comps cheaper:",
      (above["pct_comps_cheaper"] < 40).sum())

below = comps[comps["gap_vs_median_pct"] < 0]
print("Below median but more than 60% of comps cheaper:",
      (below["pct_comps_cheaper"] > 60).sum())

count    21514.0
mean        37.1
std        225.5
min        -96.5
5%         -50.0
25%        -21.3
50%          1.7
75%         42.3
95%        197.1
max      16533.7
Name: gap_vs_median_pct, dtype: float64

Above median but fewer than 40% of comps cheaper: 0
Below median but more than 60% of comps cheaper: 0


In [3]:
# Manual check of one listing
row = comps.sample(1, random_state=7).iloc[0]
me = prep[prep["listing_id"] == row["listing_id"]].iloc[0]

level_columns = {
    1: ["neighbourhood", "room_type", "stay_type", "bedrooms_filled"],
    2: ["neighbourhood", "room_type", "stay_type"],
    3: ["borough", "room_type", "stay_type"],
    4: ["room_type", "stay_type"],
}[int(row["comp_level"])]

# Pool: priced, not outliers, not this listing
pool = prep[
    prep["base_price"].notna()
    & (prep["price_outlier"] == False)
    & (prep["listing_id"] != me["listing_id"])
]

# Keep listings that match on every column of the chosen level
mask = pd.Series(True, index=pool.index)
for col in level_columns:
    if pd.isna(me[col]):
        mask &= pool[col].isna()          # unknown matches unknown
    else:
        mask &= pool[col] == me[col]
group = pool[mask]

print(me[["listing_id", "neighbourhood", "room_type", "stay_type",
          "bedrooms_filled", "base_price"]])
print()
print(f"Level used: {int(row['comp_level'])}")
print(f"Comp count     SQL: {int(row['comp_count']):>8}   pandas: {len(group):>8}")
print(f"Median         SQL: {row['comp_median']:>8.2f}   pandas: {group['base_price'].median():>8.2f}")
print(f"% cheaper      SQL: {row['pct_comps_cheaper']:>8.1f}   "
      f"pandas: {(group['base_price'] < me['base_price']).mean() * 100:>8.1f}")

listing_id         925243479022156915
neighbourhood            Williamsburg
room_type             Entire home/apt
stay_type                monthly stay
bedrooms_filled                   1.0
base_price                 538.366667
Name: 9050, dtype: object

Level used: 1
Comp count     SQL:      219   pandas:      219
Median         SQL:   227.49   pandas:   227.49
% cheaper      SQL:     95.0   pandas:     95.0


In [4]:
# Which levels are used, by segment?
merged = comps.merge(prep[["listing_id", "room_type", "stay_type"]], on="listing_id")
print(pd.crosstab([merged["room_type"], merged["stay_type"]], merged["comp_level"]))

comp_level                       1     2    3  4
room_type       stay_type                       
Entire home/apt monthly stay  8104  1383  460  0
                short stay     828   519  237  5
Hotel room      monthly stay     0     0   20  1
                short stay     320    87   52  0
Private room    monthly stay  5144   857  400  0
                short stay    2000   470  435  0
Shared room     monthly stay    37    28   84  3
                short stay       0     0   31  9
